In [ ]:
import json
import requests
from urllib.request import urlopen

# CONFIG
NOTEBOOK_PATH = "example_notebook.ipynb"  # or raw GitHub URL if you want to fetch it directly
API_URL = "https://service.tib.eu/sandbox/nfdi4energyannotator/annotate"

# --- Helper to call annotation API ---
def annotate_text(text, ontology_ids=["oeo"], max_depth=0):
    payload = {"text": text, "ontology_ids": ontology_ids, "max_depth": max_depth}
    headers = {"accept": "application/json", "Content-Type": "application/json"}
    resp = requests.post(API_URL, headers=headers, json=payload)
    resp.raise_for_status()
    return resp.json().get("matches", [])

# --- Helper to highlight terms ---
def highlight_text(text, matches):
    # Sort matches by start index (descending to avoid index shift during insertion)
    matches = sorted(matches, key=lambda m: m["start"], reverse=True)
    for m in matches:
        start, end = m["start"], m["end"]
        iri = m["iri"]
        label = m["label"]
        term = text[start:end]
        link = f'<a href="{iri}" target="_blank" style="background-color:#ffff99;text-decoration:none;color:inherit;">{term}</a>'
        text = text[:start] + link + text[end:]
    return text

# --- Load notebook ---
with open(NOTEBOOK_PATH, "r", encoding="utf-8") as f:
    notebook = json.load(f)

# --- Annotate markdown cells ---
for cell in notebook["cells"]:
    if cell["cell_type"] == "markdown":
        text = "".join(cell["source"])
        matches = annotate_text(text)
        annotated_text = highlight_text(text, matches)
        # Replace cell source with HTML-friendly markdown
        cell["source"] = [annotated_text]

# --- Save updated notebook (overwrite) ---
with open(NOTEBOOK_PATH, "w", encoding="utf-8") as f:
    json.dump(notebook, f, indent=2)

print(f"✅ Annotated notebook saved: {NOTEBOOK_PATH}")
